In [ ]:
<span style="font-size: 17px;">

```python
import pandas as pd 
path = R"../data/HIWIN_Specs.xlsx"

# 讀取各分頁
FSV = pd.read_excel(path, sheet_name = "FSV")
FSW = pd.read_excel(path, sheet_name = "FSW")
FSI = pd.read_excel(path, sheet_name = "FSI")
RSI = pd.read_excel(path, sheet_name = "RSI")

sheet = [FSV, FSW, FSI, RSI]
sheet_name = ["FSV", "FSW", "FSI", "RSI"]

# 合併數據
all = pd.concat(sheet, axis = 0)

# 寫入 Excel
with pd.ExcelWriter(path, engine = 'openpyxl', mode = 'a', if_sheet_exists = 'replace') as writer:
    for i in range(len(sheet)):
        sheet[i].to_excel(writer, sheet_name = sheet_name[i], index = False)
    all.to_excel(writer, sheet_name = 'ALL', index = False)

In [2]:
### 指定輸入參數
maximum_feed_rate = 48000 #最大進給速率, mm/min
motor_max_speed = 4000 #馬達最高轉速，預設3000 rpm
acceleration = "" #加速度
reduction_ratio = 1 #減速比
load = 775 #負載
cutting_force = 343 #切削力
length = 924 # 螺桿長度，兩端軸承間距
preload_rate = 0.05 #預壓率
axis = ["x", "y", "z"]
gravity_axis_YN = True #判斷重力軸
guide = maximum_feed_rate / (motor_max_speed * reduction_ratio)
N = maximum_feed_rate / guide #螺桿最高轉速

from math import pi

# 導程 & 最大轉速，最大進給速率 = 導程 * 最大轉速 * 減速比
guide = maximum_feed_rate / (motor_max_speed * reduction_ratio) #導程

#直徑計算
def Diameter_calculation():
    Nm = (maximum_feed_rate / guide) * 0.5 #臨界轉速

    #由導螺桿臨界轉速估算導螺桿桿徑
    f = [9.7, 15.1, 21.9, 3.4] #[支-支, 固-支, 固-固, 固-自]
    dr_n = round((Nm * length**2 / f[2]) * 1e-7, 0) # dr = (n * (length**2) / f) * (10**-7)

    #由挫曲負荷估算導螺桿桿徑
    if gravity_axis_YN:
        p = (load + cutting_force) * 2
    else:
       cof = 0.008 #摩擦力係數
       ff = load * cof 
       p = (cutting_force + ff) * 2
    E = 21000 #kgf/mm2
    n = [4.0, 2.0, 0.25] #[固-支, 固-固, 固-自]
    dr_p = round((p * 64 * (length**2) / (n[0] * (pi**3) * E))**0.25, 0)
    #取大值
    print(f"由挫曲負荷估算導螺桿桿徑: {dr_p}mm, 由導螺桿臨界轉速估算導螺桿桿徑: {dr_n}mm")
    dr_F =  max(dr_n, dr_p)
    #由DN估算導螺桿桿徑
    dr_DN = round(150000 / N, 0)
    print(f"直徑下限: {dr_F}mm, 直徑上限: {dr_DN}mm")
    print(f"{dr_F}mm < 螺桿直徑 < {dr_DN}mm")

    d_list = [12, 14, 15, 16, 20, 25, 28, 32, 36, 40, 45, 50, 55, 63, 70, 80, 100]
    suitable_dr = []
    cunt = 0
    found_any = False
    for diameter in range(len(d_list)):
        # 同時符合強度要求 (dr_F) 且在轉速限制內 (dr_DN)
        
        if d_list[diameter] >= dr_F and d_list[diameter] <= dr_DN:
            suitable_dr.append(d_list[diameter])
            cunt = diameter
            found_any = True
    if found_any and cunt+1 < len(d_list):
        suitable_dr.append(d_list[cunt+1])
    print(suitable_dr)

    return dr_F, dr_DN, suitable_dr
dr_F, dr_DN, suitable_dr = Diameter_calculation()
print("="*100)
print(f"導程: {guide}")
print("="*100)

#動負荷計算
def C_calculation():
    
    if gravity_axis_YN:
            p = (load + cutting_force)
    else:
        cof = 0.008 #摩擦力係數
        ff = load * cof 
        p = (cutting_force + ff)

    c = round(p / 3 / preload_rate, 0)
    print(f"動負荷: {c} kfg")  
    return c
c = C_calculation()

由挫曲負荷估算導螺桿桿徑: 15.0mm, 由導螺桿臨界轉速估算導螺桿桿徑: 8.0mm
直徑下限: 15.0mm, 直徑上限: 38.0mm
15.0mm < 螺桿直徑 < 38.0mm
[15, 16, 20, 25, 28, 32, 36, 40]
導程: 12.0
動負荷: 7453.0 kfg


In [8]:
# suitable_dr = 螺桿直徑列表
# guide = 導程
# c = 動負荷
import pandas as pd

par = {"公稱 外徑":suitable_dr, 
       "導程": guide, 
       "動負荷 C (kfg)": 7453.0
       }

path = R"../data/HIWIN_Specs.xlsx"
fsv = pd.read_excel(path, sheet_name = "FSV")
fsw = pd.read_excel(path, sheet_name = "FSW")
fsi = pd.read_excel(path, sheet_name = "FSI")
rsi = pd.read_excel(path, sheet_name = "RSI")
all = pd.read_excel(path, sheet_name = "ALL")

## 導程 --> 外徑 --> 動負荷

import pandas as pd

# 1. 取得資料庫中所有可用的導程，並計算與目標的距離
# sorted(key=...) 會讓距離 12 最近的排在前面（例如 10, 16, 20...）
available_guides = sorted(all["導程"].unique(), key = lambda x: abs(x - guide))

# 限制搜尋範圍：只找最接近的前 4 個導程等級（避免找太遠的規格）
search_guides = available_guides[:4] 

print(f"預計嘗試的導程順序(最接近前4名): {search_guides}\n")

found = False

# 2. 開始階層式搜尋
for current_g in search_guides:
    # 步驟一：過濾當前導程
    df_step1 = all.loc[all["導程"] == current_g]
    
    # 步驟二：過濾外徑清單
    if not df_step1.empty:
        df_step2 = df_step1.loc[df_step1["公稱 外徑"].isin(par["公稱 外徑"])]
        
        # 步驟三：過濾動負荷 (>= 目標值)
        if not df_step2.empty:
            df_step3 = df_step2.loc[df_step2["動負荷 C (kfg)"] >= par["動負荷 C (kfg)"]]
            
            if not df_step3.empty:
                # 排序：動負荷越高越好 (壽命較長)
                best_match = df_step3.sort_values(by="動負荷 C (kfg)", ascending=False)
                
                print(f"在導程 {current_g} 找到符合型號！(與目標差距: {abs(current_g - guide)})")
                display(best_match[["型號", "公稱 外徑", "導程", "動負荷 C (kfg)"]].head())
                
                best_choice = best_match.iloc[0]
                print(f"最終推薦：{best_choice['型號']}")
                found = True
                break # 找到最接近且達標的就跳出迴圈
            else:
                print(f"導程 {current_g} 雖然符合外徑，但動負荷皆小於 {par['動負荷 C (kfg)']}。")
        else:
            print(f"導程 {current_g} 的清單中沒有你指定的外徑 {par['公稱 外徑']}。")

if not found:
    print(f"\n在最接近的兩個導程 {search_guides} 中，仍找不到符合外徑與動負荷條件的螺桿。")

預計嘗試的導程順序(最接近前4名): [np.float64(12.0), np.float64(10.0), np.float64(15.0), np.float64(8.0)]

導程 12.0 雖然符合外徑，但動負荷皆小於 7453.0。
導程 10.0 雖然符合外徑，但動負荷皆小於 7453.0。
導程 15.0 的清單中沒有你指定的外徑 [15, 16, 20, 25, 28, 32, 36, 40]。
導程 8.0 雖然符合外徑，但動負荷皆小於 7453.0。

在最接近的兩個導程 [np.float64(12.0), np.float64(10.0), np.float64(15.0), np.float64(8.0)] 中，仍找不到符合外徑與動負荷條件的螺桿。


In [9]:
def diagnose_selection(all_df, target_guide, par):
    print("--- 選型失效診斷報告 ---")
    
    # 1. 檢查導程分佈
    guides = sorted(all_df["導程"].unique())
    print(f"1. 目前資料庫導程有: {guides}")
    
    # 2. 檢查在目標導程附近，最強的負荷是多少
    nearby_guides = [g for g in guides if abs(g - target_guide) <= 5]
    for g in nearby_guides:
        max_c = all_df[all_df["導程"] == g]["動負荷 C (kfg)"].max()
        print(f"   -> 導程 {g} 的最大負荷僅為 {max_c} kgf")
    
    # 3. 檢查外徑限制
    available_dr = all_df[all_df["動負荷 C (kfg)"] >= par["動負荷 C (kfg)"]]["公稱 外徑"].unique()
    print(f"2. 若要滿足負荷 {par['動負荷 C (kfg)']}，外徑至少需要: {sorted(available_dr)}")
    
    if not any(dr in par["公稱 外徑"] for dr in available_dr):
        print("結論：你的外徑清單太小，無法承載這麼高的負荷。")

diagnose_selection(all, 12, par)

--- 選型失效診斷報告 ---
1. 目前資料庫導程有: [np.float64(2.0), np.float64(2.5), np.float64(2.54), np.float64(4.0), np.float64(5.0), np.float64(5.08), np.float64(6.0), np.float64(8.0), np.float64(10.0), np.float64(12.0), np.float64(15.0), np.float64(16.0), np.float64(20.0), np.float64(25.0), np.float64(30.0), np.float64(32.0), np.float64(35.0), np.float64(40.0), np.float64(50.0)]
   -> 導程 8.0 的最大負荷僅為 5674 kgf
   -> 導程 10.0 的最大負荷僅為 10207 kgf
   -> 導程 12.0 的最大負荷僅為 15251 kgf
   -> 導程 15.0 的最大負荷僅為 7030 kgf
   -> 導程 16.0 的最大負荷僅為 25684 kgf
2. 若要滿足負荷 7453.0，外徑至少需要: [np.int64(45), np.int64(50), np.int64(55), np.int64(60), np.int64(63), np.int64(70), np.int64(80), np.int64(100)]
結論：你的外徑清單太小，無法承載這麼高的負荷。
